# VLA 연구 · 13회차 — 숫자 조건 VLM → VLA 엔드투엔드 파이프라인

| 항목 | 내용 |
|---|---|
| 오늘 목표 | 숫자가 찍힌 **동일한 정육면체**를 VLM이 읽고 VLA가 옮기는 파이프라인을 끝까지 돌린다 |
| 핵심 질문 | 색·모양 단서를 없앴을 때 **1단계(VLM)와 2단계(VLA)의 성능이 각각 얼마나 떨어지는가?** |
| 환경 | MuJoCo + Franka Panda, 관측·액션은 **LIBERO 호환** 규격 (12회차와 동일) |
| 산출물 | 색 vs 숫자 비교표, 오류 분석(지각/추론 분리), 숫자 조건 rollout GIF, 데이터셋 준비 |

### 12회차와 달라진 점

12회차에서는 **색과 모양**이 서로 다른 10개 물체를 다뤘습니다.
VLM은 "빨간 블록"처럼 색만 보면 물체를 구분할 수 있었습니다.

이번에는 물체 10개가 **전부 똑같은 흰 정육면체**이고, 면에 찍힌 **숫자(1~10)**만이
유일한 구분 단서입니다. 색도 크기도 모양도 같으므로:

- VLM이 숫자를 **읽지 못하면** 계획을 세울 수 없고
- VLA가 지시문의 숫자를 **알아보지 못하면** 어떤 큐브를 집어야 하는지 모릅니다

이 조건에서 성능이 떨어진다면 그건 "옮기는 게 어려워서"가 아니라
**"무엇을 옮길지 알아보지 못해서"** 입니다.

---
## SECTION 0 | 설치와 코드 연결

12회차와 동일합니다. `vla_pipeline` 폴더가 팀 공유 드라이브에 있어야 합니다.

```
MyDrive/VLA_Team_2026/
└── vla_pipeline/          <- make_student_zip.py 로 만든 zip 을 푼 것
    ├── vlapipe/  scripts/  assets/  notebooks/
    └── README.md ...
```

In [ ]:
# 0-1. 패키지 설치 (Colab 최초 1회, 이후 런타임 재시작)
!pip install -q 'mujoco>=3.2,<4.0' h5py imageio imageio-ffmpeg 'transformers>=4.50' accelerate num2words torchvision Pillow
print('설치 완료 — Runtime > Restart session 후 0-2 부터 실행하세요.')

In [ ]:
# 0-2. Drive 마운트 + 코드를 런타임 로컬로 복사
import os, sys, shutil
os.environ['MUJOCO_GL'] = 'egl'
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    TEAM = Path('/content/drive/MyDrive/VLA_Team_2026')
    SRC = TEAM / 'vla_pipeline'
    assert SRC.exists(), f'공유 드라이브에 없습니다: {SRC}'
    PKG = Path('/content/vla_pipeline')
    shutil.rmtree(PKG / 'data', ignore_errors=True)
    shutil.copytree(SRC, PKG, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('.venv', 'outputs', '__pycache__', '.git'))
    print('코드 사본 :', PKG)
    print('원본      :', SRC, '(읽기만 함)')
else:
    PKG = Path.cwd().parent
    TEAM = PKG.parent

assert PKG.exists(), f'경로를 찾을 수 없습니다: {PKG}'
sys.path.insert(0, str(PKG))
os.chdir(PKG)
(PKG / 'outputs').mkdir(exist_ok=True)

n_ep = len(list((PKG / 'data' / 'episodes').glob('*.hdf5'))) if (PKG / 'data' / 'episodes').exists() else 0
print('작업 폴더 :', PKG)
print('녹화본    :', n_ep, '개')

In [ ]:
# 0-3. Panda 에셋 준비 (최초 1회, 약 40MB)
!python setup_assets.py

In [ ]:
# 0-4. 환경 점검
!python scripts/check_env.py

---
## SECTION 1 | 숫자 조건 씬 살펴보기

색 조건과 숫자 조건의 차이를 눈으로 확인합니다.

| | 색 조건 (`color`) | 숫자 조건 (`number`) |
|---|---|---|
| 물체 | 색·크기·모양이 다른 10개 | **전부 똑같은 흰 정육면체** 10개 |
| 구분 단서 | 색 + 모양 | 면에 찍힌 **숫자 1~10** 뿐 |
| 정돈 규칙 | 긴 블록→오른쪽 / 정육면체·타일→왼쪽 | **1~5→오른쪽 / 6~10→왼쪽** |
| 지시문 | "Pick up the red block …" | "Pick up the cube number 7 …" |

**바뀌지 않는 것**: 테이블·구역·조명·카메라·로봇, 배치, LIBERO 규격.

In [ ]:
# 1-1. 색 조건과 숫자 조건을 나란히 렌더
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from vlapipe.env import SortingEnv
from vlapipe import config as C

# --- 색 조건 ---
env_color = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)
env_color.reset(layout='messy', seed=3)
img_color_256 = env_color.render('agentview', 256, 256)
img_color_512 = env_color.render('agentview', 512, 512)

# --- 숫자 조건 ---
from vlapipe import numbered
numbered.activate()
env_num = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)
env_num.reset(layout='messy', seed=3)
img_num_256 = env_num.render('agentview', 256, 256)
img_num_512 = env_num.render('agentview', 512, 512)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes[0, 0].imshow(img_color_256); axes[0, 0].set_title('Color  256px', fontsize=12); axes[0, 0].axis('off')
axes[0, 1].imshow(img_color_512); axes[0, 1].set_title('Color  512px', fontsize=12); axes[0, 1].axis('off')
axes[1, 0].imshow(img_num_256);   axes[1, 0].set_title(f'Number 256px  (face ~{numbered.face_pixels(256):.1f} px)', fontsize=12); axes[1, 0].axis('off')
axes[1, 1].imshow(img_num_512);   axes[1, 1].set_title(f'Number 512px  (face ~{numbered.face_pixels(512):.1f} px)', fontsize=12); axes[1, 1].axis('off')
plt.suptitle('Color condition (top) vs Number condition (bottom) — same seed, same layout', fontsize=13)
plt.tight_layout(); plt.show()

print('물체 세트:', C.ACTIVE_OBJECT_SET)
print('물체 이름:', C.OBJECT_NAMES)
print('정돈 규칙: 1~%d -> 오른쪽, %d~%d -> 왼쪽' % (C.NUMBER_SPLIT, C.NUMBER_SPLIT + 1, C.NUMBER_COUNT))
print('잘못 놓인:', env_num.misplaced_objects())
print()
print('위 그림에서 256px 숫자 조건(왼쪽 아래)의 숫자를 읽어 보세요.')
print('읽히지 않는다면, VLM 에게 256px 이미지를 주는 것은 숫자를 지우고 구분해 보라는 것과 같습니다.')

In [ ]:
# 1-2. 숫자 조건 — 카메라 4대로 보기
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, cam in zip(axes, C.ALL_CAMERAS):
    ax.imshow(env_num.render(cam, 400, 400))
    ax.set_title(cam, fontsize=10); ax.axis('off')
plt.suptitle('Number condition — all cameras (seed=3, messy)', fontsize=12)
plt.tight_layout(); plt.show()

print('agentview(top-down)에서 숫자가 가장 잘 보입니다.')
print('robot0_eye_in_hand(손목)에서도 가까이 가면 보이지만, 멀리서는 구분이 안 됩니다.')

---
## SECTION 2 | 1단계 — VLM 플래너로 숫자 읽기

숫자 조건의 핵심은 **VLM이 이미지에서 숫자를 읽을 수 있는가** 입니다.

두 가지 프롬프트를 비교합니다:

| 프롬프트 | VLM이 하는 일 | 규칙(1~5/6~10)은 누가 적용? |
|---|---|---|
| `number` | 숫자를 읽고 **목적지까지** 말한다 | VLM |
| `number_read` | 가운데 있는 숫자만 읽는다 | **코드** |

**읽는 법:**
- `number_read`도 낮다 → **숫자를 못 읽는 것** (지각 문제)
- `number_read`는 높은데 `number`가 낮다 → **읽기는 되는데 규칙을 못 쓰는 것** (추론 문제)
- 둘 다 색 조건보다 많이 낮다 → 색 조건의 점수에 **단서가 쉬워서 얻은 점수**가 섞여 있었다

In [ ]:
# 2-1. 정답 계획(규칙 플래너) 확인
from vlapipe.planner import RulePlanner, VLMPlanner, plan_accuracy, parse_plan, PROMPTS

env_num.reset(layout='messy', seed=3)
gold = RulePlanner(env_num).plan('테이블을 정리하시오')

print('=== 정답 계획 (규칙 플래너) ===')
for s in gold:
    print(f"  {s['step']}. {s['instruction']}")
    print(f"     {s['instruction_ko']}   -> ({s['object']}, {s['target_zone']})")
print(f'\n잘못 놓인 물체 {len(gold)}개')

In [ ]:
# 2-2. VLM(SmolVLM2)으로 숫자를 읽고 계획을 세우게 한다
VLM_PX = 512   # 숫자 조건은 512px 로 렌더해야 숫자가 보인다 (1-1 에서 확인)

vlm = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct', prompt_style='number')

env_num.reset(layout='messy', seed=3)
img = env_num.render('agentview', VLM_PX, VLM_PX)
raw = vlm.raw_response(img, '테이블을 정리하시오')

print('=== VLM 원문 (prompt_style="number") ===')
print(raw)

pred = parse_plan(raw)
print('\n=== 파싱된 계획 ===')
for s in pred:
    print(f"  {s['step']}. {s['object']} -> {s['target_zone']}")

print('\n=== 채점 ===')
acc = plan_accuracy(pred, gold)
for k, v in acc.items():
    print(f'  {k:14s} {v}')

In [ ]:
# 2-3. number_read 프롬프트 — 숫자만 읽게 하고 목적지는 코드가 결정
vlm.prompt_style = 'number_read'
env_num.reset(layout='messy', seed=3)
raw_read = vlm.raw_response(env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오')

print('=== VLM 원문 (prompt_style="number_read") ===')
print(raw_read)

pred_read = parse_plan(raw_read)
acc_read = plan_accuracy(pred_read, gold)
print('\n=== 채점 ===')
for k, v in acc_read.items():
    print(f'  {k:14s} {v}')

In [ ]:
# 2-4. 채점 루프 — 여러 시드 × 프롬프트 두 가지
SEEDS = [3, 7, 11, 19, 23]        # 수업 중에는 5개 (T4 에서 몇 분)
# SEEDS = list(range(30))         # 보고서용: 조건당 30씬

rows_num = []
for style in ['number', 'number_read']:
    vlm.prompt_style = style
    for seed in SEEDS:
        env_num.reset(layout='messy', seed=seed)
        g = RulePlanner(env_num).plan()
        p = parse_plan(vlm.raw_response(
            env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'))
        a = plan_accuracy(p, g)
        rows_num.append({'프롬프트': style, 'seed': seed,
                         '정답수': a['n_gold'], '예측수': a['n_pred'], '맞춘수': a['correct'],
                         'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                         '빠뜨림': len(a['missed']), '없는물체': len(a['hallucinated']),
                         '구역틀림': len(a['wrong_zone'])})

df_num = pd.DataFrame(rows_num)
display(df_num)

In [ ]:
# 2-5. 색 조건도 같은 해상도(512px)로 다시 재기 — 공정한 비교
numbered.deactivate()
vlm.prompt_style = 'json'

rows_col = []
for seed in SEEDS:
    env_color.reset(layout='messy', seed=seed)
    g = RulePlanner(env_color).plan()
    p = parse_plan(vlm.raw_response(
        env_color.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'))
    a = plan_accuracy(p, g)
    rows_col.append({'프롬프트': 'json (color)', 'seed': seed,
                     '정답수': a['n_gold'], '예측수': a['n_pred'], '맞춘수': a['correct'],
                     'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                     '빠뜨림': len(a['missed']), '없는물체': len(a['hallucinated']),
                     '구역틀림': len(a['wrong_zone'])})

df_col = pd.DataFrame(rows_col)

# 숫자 조건으로 다시 돌아간다 (이후 SECTION 에서 숫자 조건을 쓰므로)
numbered.activate()
env_num = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)

In [ ]:
# 2-6. 세 조건 비교 요약표 (micro 평균)
def micro(df, label):
    c, g, n = int(df['맞춘수'].sum()), int(df['정답수'].sum()), int(df['예측수'].sum())
    return {'조건': label, '씬수': len(df), '정답수': g, '예측수': n, '맞춘수': c,
            'recall': round(c / g, 2) if g else float('nan'),
            'precision': round(c / n, 2) if n else float('nan'),
            '빠뜨림': int(df['빠뜨림'].sum()), '없는물체': int(df['없는물체'].sum()),
            '구역틀림': int(df['구역틀림'].sum())}

summary = pd.DataFrame([
    micro(df_col, 'A. 색 (json, 512px)'),
    micro(df_num[df_num['프롬프트'] == 'number'], 'B. 숫자 (number, 512px)'),
    micro(df_num[df_num['프롬프트'] == 'number_read'], 'C. 숫자 (number_read, 512px)'),
])
display(summary)

print('\n읽는 법:')
print('  A > B ≈ C  →  색 단서가 쉬워서 A의 점수가 높았다. 숫자 읽기 자체가 병목.')
print('  A > B > C  →  숫자를 읽지 못하는 문제 + 규칙 적용 문제가 겹쳐 있다.')
print('  A ≈ B ≈ C  →  구분 단서와 무관하게 계획 능력이 동일하다 (가능성 낮음).')
print('  C > B      →  숫자는 읽지만 규칙 적용에서 틀린다.')
print()
print('※ 씬 %d개 기준입니다. 보고서에는 SEEDS 를 30개로 늘려서 재세요.' % len(SEEDS))

---
## SECTION 3 | 해상도 실험 — 256px vs 512px

숫자 조건에서 **이미지 해상도**가 VLM 성능에 얼마나 영향을 주는지 확인합니다.

- 색은 한 픽셀만 봐도 알 수 있다 → 해상도에 둔감
- 숫자는 글자 모양을 읽어야 한다 → 해상도가 낮으면 읽을 수 없다

이 차이가 실측으로 확인되면, "숫자를 읽는 것이 이 과제의 병목"이라는 주장의 직접적 근거가 됩니다.

In [ ]:
# 3-1. 숫자 조건을 256px / 512px 로 나란히 채점
rows_res = []
for px in [256, 512]:
    vlm.prompt_style = 'number'
    for seed in SEEDS:
        env_num.reset(layout='messy', seed=seed)
        g = RulePlanner(env_num).plan()
        p = parse_plan(vlm.raw_response(
            env_num.render('agentview', px, px), '테이블을 정리하시오'))
        a = plan_accuracy(p, g)
        rows_res.append({'해상도': f'{px}px', 'seed': seed,
                         'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                         '맞춘수': a['correct'], '정답수': a['n_gold'], '예측수': a['n_pred']})

df_res = pd.DataFrame(rows_res)

def micro_res(df, label):
    c, g, n = int(df['맞춘수'].sum()), int(df['정답수'].sum()), int(df['예측수'].sum())
    return {'조건': label, 'recall': round(c/g, 2) if g else 0, 'precision': round(c/n, 2) if n else 0}

display(pd.DataFrame([micro_res(df_res[df_res['해상도']==r], f'숫자 ({r})') for r in ['256px', '512px']]))

print('\n해상도를 반으로 줄이면 점수가 얼마나 떨어지나요?')
print('많이 떨어진다면 → VLM이 숫자를 읽는 것이 병목입니다.')
print('비슷하다면 → 어차피 숫자를 못 읽고 있으므로 해상도와 무관합니다.')

---
## SECTION 4 | 2단계 — VLA 정책으로 숫자 큐브 옮기기

1단계(VLM)가 계획을 세우면, 2단계(VLA)가 실제로 물체를 집어 옮깁니다.

여기서는 먼저 **규칙 플래너(정답)** + **ScriptedSortingPolicy(teacher)**로 돌려
숫자 조건에서 제어 자체가 잘 되는지 확인합니다.

색 조건과 비교해서 **제어 성능이 비슷하면** → 1단계(VLM)만의 문제라고 말할 수 있습니다.

In [ ]:
# 4-1. 숫자 조건 — 규칙 플래너 + teacher 정책으로 rollout
from vlapipe.policy import ScriptedSortingPolicy
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display as ipy_display

env_num.reset(layout='messy', seed=3)
policy = ScriptedSortingPolicy(env_num)
plan = RulePlanner(env_num).plan('테이블을 정리하시오')

frames, log = [], []
total_steps = 0
for st in plan:
    policy.set_subtask(st); n = 0
    while not policy.done and n < 500:
        env_num.step(policy.select_action()); n += 1
        if n % 3 == 0:
            frames.append(env_num.render('frontview', 300, 400))
    total_steps += n
    ok = env_num.zone_of(st['object']) == st['target_zone']
    log.append({'물체': st['object'], '스텝': n, '성공': ok, '실패사유': policy.failure})

df_log = pd.DataFrame(log)
display(df_log)
print(f'\n정리 완료: {env_num.is_tidy()} | 총 스텝: {total_steps}')
print(f'남은 물체: {env_num.misplaced_objects()}')

imageio.mimsave('outputs/section4_number.gif', frames[::2], fps=14, loop=0)
ipy_display(IPyImage(filename='outputs/section4_number.gif'))

In [ ]:
# 4-2. 여러 시드에서 숫자 조건의 teacher 성공률
results_teacher = []
for seed in SEEDS:
    env_num.reset(layout='messy', seed=seed)
    pol = ScriptedSortingPolicy(env_num)
    pln = RulePlanner(env_num).plan()
    ok_count, fail_count = 0, 0
    steps = 0
    for st in pln:
        pol.set_subtask(st); n = 0
        while not pol.done and n < 500:
            env_num.step(pol.select_action()); n += 1
        steps += n
        if env_num.zone_of(st['object']) == st['target_zone']:
            ok_count += 1
        else:
            fail_count += 1
    results_teacher.append({
        'seed': seed, '계획수': len(pln), '성공': ok_count, '실패': fail_count,
        '스텝': steps, '정리완료': env_num.is_tidy()})

df_teacher = pd.DataFrame(results_teacher)
display(df_teacher)

total_ok = df_teacher['성공'].sum()
total_all = (df_teacher['성공'] + df_teacher['실패']).sum()
print(f'\nteacher 운반 성공률: {total_ok}/{total_all} ({total_ok/total_all*100:.0f}%)')
print(f'씬 정리 완료: {df_teacher["정리완료"].sum()}/{len(df_teacher)}')
print('\n이 결과가 색 조건(12회차)과 비슷하면 → 제어는 문제없고 1단계(VLM)만 병목.')

---
## SECTION 5 | 엔드투엔드 — VLM 계획 + VLA 실행

지금까지는 1단계(VLM)와 2단계(VLA)를 따로따로 봤습니다.
이제 **진짜 파이프라인**: VLM이 이미지를 보고 계획을 세우고, VLA가 실행합니다.

```
이미지(숫자 큐브) + "테이블을 정리하시오"
  │
  ├─ [1단계 VLM]  SmolVLM2 : 숫자를 읽고 이동 계획 생성
  │                         "Pick up the cube number 7 and place it in the left storage zone."
  └─ [2단계 VLA]  Teacher  : 지시문에 따라 관절 제어
```

In [ ]:
# 5-1. 숫자 조건 엔드투엔드 (VLM 플래너 + teacher 정책)
!python scripts/run_pipeline.py --objects number --layout messy --seed 3 --passes 2 --planner vlm

In [ ]:
# 5-2. 같은 시드에서 규칙 플래너(정답)로도 돌려 비교
!python scripts/run_pipeline.py --objects number --layout messy --seed 3 --passes 2 --planner rule

In [ ]:
# 5-3. 여러 시드에서 VLM 플래너 엔드투엔드 성능
vlm.prompt_style = 'number'

results_e2e = []
for seed in SEEDS:
    env_num.reset(layout='messy', seed=seed)
    img_vlm = env_num.render('agentview', VLM_PX, VLM_PX)
    raw_plan = vlm.raw_response(img_vlm, '테이블을 정리하시오')
    vlm_plan = parse_plan(raw_plan)
    gold_plan = RulePlanner(env_num).plan()
    acc_e2e = plan_accuracy(vlm_plan, gold_plan)

    pol = ScriptedSortingPolicy(env_num)
    ok_count, fail_count, steps = 0, 0, 0
    for st in vlm_plan:
        pol.set_subtask(st); n = 0
        while not pol.done and n < 500:
            env_num.step(pol.select_action()); n += 1
        steps += n
        if env_num.zone_of(st['object']) == st['target_zone']:
            ok_count += 1
        else:
            fail_count += 1

    results_e2e.append({
        'seed': seed, 'VLM계획수': len(vlm_plan), '정답수': len(gold_plan),
        'plan_recall': round(acc_e2e['recall'], 2),
        '운반성공': ok_count, '운반실패': fail_count,
        '스텝': steps, '정리완료': env_num.is_tidy(),
        '남은물체': len(env_num.misplaced_objects())})

df_e2e = pd.DataFrame(results_e2e)
display(df_e2e)

print(f'\n평균 plan recall: {df_e2e["plan_recall"].mean():.2f}')
print(f'씬 정리 완료: {df_e2e["정리완료"].sum()}/{len(df_e2e)}')

---
## SECTION 6 | 오류 분석 — 어디서 틀리는가

엔드투엔드에서 실패했을 때 **어디서 틀렸는지**를 분리합니다.

1. **VLM이 틀린 것**: 없는 물체를 옮기라고 함(환각), 빠뜨림, 구역 틀림
2. **VLA가 틀린 것**: 물체를 못 잡음, 이송 중 떨어뜨림
3. **해상도 때문인 것**: 512px→256px 로 바꿨을 때 VLM 점수가 떨어지면 지각 문제

이 분리가 되어야 "무엇을 고쳐야 하는가"에 답할 수 있습니다.

In [ ]:
# 6-1. VLM 오류 유형 집계
print('=== VLM 오류 유형 (숫자 조건, number 프롬프트) ===')
num_df = df_num[df_num['프롬프트'] == 'number']
print(f'  빠뜨림 (정답에 있는데 말하지 않음) : {num_df["빠뜨림"].sum()}')
print(f'  없는물체 (옮길 필요 없는 것을 옮기라 함) : {num_df["없는물체"].sum()}')
print(f'  구역틀림 (물체는 맞고 목적지가 틀림) : {num_df["구역틀림"].sum()}')
print()

print('=== VLM 오류 유형 (색 조건, json 프롬프트) ===')
print(f'  빠뜨림 : {df_col["빠뜨림"].sum()}')
print(f'  없는물체 : {df_col["없는물체"].sum()}')
print(f'  구역틀림 : {df_col["구역틀림"].sum()}')
print()

print('숫자 조건의 "없는물체"가 많다면 → VLM이 숫자를 아무거나 찍고 있다 (읽지 않음)')
print('숫자 조건의 "구역틀림"이 많다면 → 숫자는 읽지만 1~5/6~10 규칙을 모른다')
print('숫자 조건의 "빠뜨림"이 많다면 → 숫자를 보는 것 자체가 안 된다')

In [ ]:
# 6-2. VLM이 특정 숫자를 잘/못 읽는지 분석
# VLM 출력에서 언급된 숫자를 추출하고, 정답과 비교
import re

mentioned_numbers = []  # VLM이 언급한 숫자들
correct_numbers = []    # 정답에 있는 숫자들

vlm.prompt_style = 'number'
for seed in SEEDS:
    env_num.reset(layout='messy', seed=seed)
    g = RulePlanner(env_num).plan()
    p = parse_plan(vlm.raw_response(
        env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'))

    # 정답에서 숫자 추출
    for st in g:
        m = re.search(r'(\d+)', st['object'])
        if m: correct_numbers.append(int(m.group(1)))

    # VLM 예측에서 숫자 추출
    for st in p:
        m = re.search(r'(\d+)', st['object'])
        if m: mentioned_numbers.append(int(m.group(1)))

from collections import Counter
pred_cnt = Counter(mentioned_numbers)
gold_cnt = Counter(correct_numbers)

print('VLM이 언급한 숫자 빈도:', dict(sorted(pred_cnt.items())))
print('정답에 있는 숫자 빈도:', dict(sorted(gold_cnt.items())))
print()

# 0~10 범위 밖의 숫자를 언급했다면 완전히 엉뚱한 것
out_of_range = [n for n in mentioned_numbers if n < 1 or n > 10]
if out_of_range:
    print(f'범위 밖 숫자 언급: {out_of_range} → VLM이 숫자를 읽지 않고 있다')
else:
    print('범위 밖 숫자 없음 (1~10 안에서 답함)')

print('\n특정 숫자 쌍(6/9, 1/7)에서 헷갈리는지 확인하세요:')
print('  6과 9를 둘 다 많이 언급 → 글자 모양 혼동 가능성')
print('  아무 숫자나 골고루 찍음 → 아예 읽지 않고 있음')

In [ ]:
# 6-3. 최종 비교 요약표
print('=' * 70)
print('  최종 비교: 색 vs 숫자 (같은 시드 · 같은 layout · 같은 모델)')
print('=' * 70)
display(summary)  # 2-6 에서 만든 표
print()
print('--- teacher 제어 (규칙 플래너 기준) ---')
display(df_teacher[['seed', '성공', '실패', '정리완료']])
print()
print('--- 엔드투엔드 (VLM + teacher) ---')
display(df_e2e[['seed', 'plan_recall', '운반성공', '운반실패', '정리완료']])
print()
print('--- 해상도 영향 ---')
display(pd.DataFrame([micro_res(df_res[df_res['해상도']==r], f'숫자 ({r})') for r in ['256px', '512px']]))
print()
print('위 표들로 다음을 판단하세요:')
print('  1) teacher 제어가 색 조건과 비슷한가? → 비슷하면 2단계는 문제 없음')
print('  2) VLM recall 이 색 > 숫자인가? → 그렇다면 1단계의 "지각" 이 병목')
print('  3) 해상도를 올리면 recall 이 오르는가? → 그렇다면 이미지 해상도가 핵심')

---
## SECTION 7 | 데이터 수집 — 숫자 조건 텔레옵 녹화 + 역재생

SmolVLA 를 숫자 조건에서 파인튜닝하려면 **숫자 큐브를 옮기는 데모 데이터**가 필요합니다.

12회차와 같은 방법: 정돈된 씬을 어지르는 쪽을 게임패드로 녹화하고, 역재생으로 정돈 데모를 만듭니다.

**녹화 명령** (윈도우 로컬에서):
```powershell
.\.\.venv\Scripts\python.exe scripts\teleop_record.py --objects number --input keyboard
```

In [ ]:
# 7-1. 역재생 변환 + 검증
!python scripts/build_dataset.py

In [ ]:
# 7-2. 역재생 결과 눈으로 확인
from vlapipe.episode import Episode
import glob

revs = sorted(glob.glob('data/episodes_reversed/*.hdf5'))
if revs:
    ep = Episode(revs[0])
    print(ep.path.name, '|', len(ep), 'frames | task:', ep.task)
    idx = np.linspace(0, len(ep) - 1, 6).astype(int)
    fig, ax = plt.subplots(1, 6, figsize=(18, 3))
    for a, i in zip(ax, idx):
        a.imshow(ep.obs['agentview_image'][i]); a.axis('off'); a.set_title(f't={i}', fontsize=8)
    plt.suptitle('Reversed tidying demo (left=start, right=end)'); plt.tight_layout(); plt.show()
    imageio.mimsave('outputs/reversed_number.gif', list(ep.obs['agentview_image'][::3]), fps=12, loop=0)
    ipy_display(IPyImage(filename='outputs/reversed_number.gif'))
else:
    print('아직 녹화 데이터가 없습니다.')
    print('윈도우에서 먼저 실행하세요:')
    print('  .\\.\\.venv\\Scripts\\python.exe scripts\\teleop_record.py --objects number --input keyboard')

---
## SECTION 8 | SmolVLA 파인튜닝 (데이터를 모은 뒤)

> 에피소드 20~50개가 모인 뒤에 실행하세요.

In [ ]:
# 8-1. LeRobot 설치
!pip install -q 'lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git'

In [ ]:
# 8-2. LeRobot 데이터셋으로 변환
!python scripts/build_dataset.py --lerobot --repo-id dshs2026/vla_number_sorting

In [ ]:
# 8-3. SmolVLA 파인튜닝
!python -m lerobot.scripts.train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=dshs2026/vla_number_sorting \
  --dataset.root=data/lerobot \
  --batch_size=8 --steps=2000 \
  --output_dir=outputs/smolvla_number_sorting \
  --job_name=vla_number_sorting --wandb.enable=false

In [ ]:
# 8-4. 학습된 SmolVLA 로 숫자 조건 파이프라인 실행
!python scripts/run_pipeline.py --objects number --policy smolvla \
  --checkpoint outputs/smolvla_number_sorting/checkpoints/last/pretrained_model \
  --layout messy --seed 3

---
## SECTION 9 | 미니과제

### 필수

#### 1. 비교표 채우기

SECTION 2~3 을 돌려 아래 표를 **실측값**으로 채우세요. 추측한 값을 적지 마세요.

| 조건 | 프롬프트 | 입력 px | 씬 수 | recall | precision | 빠뜨림 | 없는물체 | 구역틀림 |
|---|---|---|---|---|---|---|---|---|
| 색 | `json` | 512 | | | | | | |
| 숫자 | `number` | 512 | | | | | | |
| 숫자 | `number_read` | 512 | | | | | | |
| 숫자 | `number` | 256 | | | | | | |

표 아래에 **2~3문장**: 어느 쪽 문제였나? 무엇을 근거로 그렇게 보았나?

#### 2. 제어 비교

SECTION 4 의 teacher 결과를 12회차(색 조건)와 비교하세요.
- 운반 성공률이 비슷한가?
- 실패 사유(`grasp_failed`, `dropped_during_carry` 등)가 다른가?
- 비슷하다면 "제어 문제가 아니라 지각 문제"라고 말할 수 있습니다.

#### 3. 엔드투엔드 실패 분석

SECTION 5 에서 VLM + teacher 로 돌렸을 때:
- 정리 완료한 씬이 몇 개인가?
- 실패 원인이 1단계(VLM 계획 오류)인지 2단계(VLA 실행 실패)인지 적으세요.

### 생각할 질문

1. **숫자를 더 잘 읽게 하려면?** 세 가지 방법의 장단점을 비교하세요:
   - 이미지 해상도를 올린다 (512 → 768)
   - 숫자 텍스처를 더 크고 또렷하게 만든다
   - VLM 을 파인튜닝한다

2. **색 조건에서 높은 점수가 진짜 능력인가?**
   색은 한 픽셀만 봐도 구분되니까, VLM 이 이미지를 "제대로 이해"한 것이 아닐 수 있습니다.
   이 주장을 뒷받침하는 근거를 여러분의 실험 결과에서 찾으세요.

3. **실제 로봇에서 숫자 읽기는 더 어려울까?**
   시뮬레이션은 조명이 균일하고 숫자가 깨끗합니다. 실제로는?
   (힌트: 그림자, 각도, 빛 반사, 마모)

In [ ]:
# 제출용 저장
import json, shutil, datetime
TEAM_OUT = Path('/content/drive/MyDrive/VLA_Team_2026') if IN_COLAB else PKG.parent
dst = TEAM_OUT / 'results' / f'session13_{datetime.datetime.now():%m%d_%H%M}'
dst.mkdir(parents=True, exist_ok=True)

# 결과 요약 JSON 저장
results_json = {
    'vlm_comparison': summary.to_dict('records') if 'summary' in dir() else [],
    'teacher_results': df_teacher.to_dict('records') if 'df_teacher' in dir() else [],
    'e2e_results': df_e2e.to_dict('records') if 'df_e2e' in dir() else [],
}
with open('outputs/session13_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_json, f, ensure_ascii=False, indent=2, default=str)

for p in Path('outputs').glob('**/*'):
    if p.is_file() and p.suffix in {'.gif', '.png', '.json'}:
        shutil.copy(p, dst / f'{p.parent.name}_{p.name}')
print('저장:', dst)
print(sorted(x.name for x in dst.iterdir()))